In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import random
import time
import os
import sys
import copy
import math
import matplotlib.pyplot as plt
from torx.module.layer import *
from benchmark import replace_layers
import argparse
import torchvision
import torchvision.transforms as transforms
from torchvision.models import vgg16
# from torchsummary import summary
from resnet import *
from mobilenet import *
from VGG import *
from densenet import * 
import json

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.25.0
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
##Load the data 
resnet18_ec = torch.load('cifar-10/resnet18_cifar10_worst_case_error_ec.pth')
vgg16_ec = torch.load('cifar-10/vgg16_cifar10_worst_case_error_ec.pth')
densenet121_ec= torch.load('cifar-10/densenet121_cifar10_worst_case_error_ec.pth')

In [4]:
##Print the data 
# print(resnet18_ec)
# print(vgg16_ec)
# print(densenet121_ec)

##find the avg and std with respect to each key in the loaded dictionary
def avg_std(data):
    avg = {}
    std = {}
    for key in data.keys():
        avg[key] = np.mean(data[key])
        std[key] = np.std(data[key])
    return avg, std
resnet18_avg, resnet18_std = avg_std(resnet18_ec)
vgg16_avg, vgg16_std = avg_std(vgg16_ec)
densenet121_avg, densenet121_std = avg_std(densenet121_ec)
##print the avg and std
print("resnet18_avg",resnet18_avg)
print("resnet18_std",resnet18_std)
print("vgg16_avg",vgg16_avg)
print("vgg16_std",vgg16_std)
print("densenet121_avg",densenet121_avg)
print("densenet121_std",densenet121_std)


resnet18_avg {'fault_0.1': 93.75800000000001, 'fault_0.5': 93.75199999999998, 'fault_1.0': 93.76500000000001, 'fault_5.0': 93.693, 'fault_10.0': 93.651, 'fault_15.0': 93.57300000000001, 'fault_20.0': 93.534, 'fault_30.0': 93.238, 'fault_40.0': 93.23299999999999, 'fault_50.0': 93.17599999999999}
resnet18_std {'fault_0.1': 0.06539113089708708, 'fault_0.5': 0.07263607918933937, 'fault_1.0': 0.06407027391856655, 'fault_5.0': 0.15258112596255174, 'fault_10.0': 0.16585837331892403, 'fault_15.0': 0.24372320365529287, 'fault_20.0': 0.21850400453996396, 'fault_30.0': 0.29952629266894265, 'fault_40.0': 0.22248820193439597, 'fault_50.0': 0.19422667170087418}
vgg16_avg {'fault_0.1': 91.79499999999999, 'fault_0.5': 91.806, 'fault_1.0': 91.795, 'fault_5.0': 91.728, 'fault_10.0': 91.695, 'fault_15.0': 91.747, 'fault_20.0': 91.687, 'fault_30.0': 91.594, 'fault_40.0': 91.631, 'fault_50.0': 91.55600000000001}
vgg16_std {'fault_0.1': 0.06682065548915261, 'fault_0.5': 0.05919459434779533, 'fault_1.0': 0.0

In [6]:
##convert the avg dict to excel using pandas dataframe 
import pandas as pd
resnet18_avg_df = pd.DataFrame(resnet18_avg, index=[0])
vgg16_avg_df = pd.DataFrame(vgg16_avg, index=[0])
densenet121_avg_df = pd.DataFrame(densenet121_avg, index=[0])
##save as excel files
resnet18_avg_df.to_excel('cifar-10/resnet18_wc_avg.xlsx', index=False)
vgg16_avg_df.to_excel('cifar-10/vgg16_wc_avg.xlsx', index=False)
densenet121_avg_df.to_excel('cifar-10/densenet121_wc_avg.xlsx', index=False)


In [11]:
resnet18_e = torch.load('cifar-10/resnet18_cifar10_worst_case_error.pth')
# vgg16_ec = torch.load('cifar-10/vgg16_cifar10_worst_case_error_ec.pth')
densenet121_e= torch.load('cifar-10/densenet121_cifar10_worst_case_error.pth')

In [12]:
resnet18_avg, resnet18_std = avg_std(resnet18_e)
# vgg16_avg, vgg16_std = avg_std(vgg16_ec)
densenet121_avg, densenet121_std = avg_std(densenet121_e)
##print the avg and std
print("resnet18_avg",resnet18_avg)
print("resnet18_std",resnet18_std)
# print("vgg16_avg",vgg16_avg)
# print("vgg16_std",vgg16_std)
print("densenet121_avg",densenet121_avg)
print("densenet121_std",densenet121_std)

resnet18_avg {'fault_0.1': 19.741999999999997, 'fault_0.5': 10.007, 'fault_1.0': 9.7895, 'fault_5.0': 5.677, 'fault_10.0': 10.0155, 'fault_20.0': 10.0}
resnet18_std {'fault_0.1': 0.11052601503718468, 'fault_0.5': 0.05386093203798084, 'fault_1.0': 0.08930145575521153, 'fault_5.0': 0.12981140165640304, 'fault_10.0': 0.028892040426387374, 'fault_20.0': 0.0}
densenet121_avg {'fault_0.1': 41.180499999999995, 'fault_0.5': 11.3325, 'fault_1.0': 9.9895, 'fault_5.0': 9.6075, 'fault_10.0': 9.921499999999998, 'fault_20.0': 9.9655}
densenet121_std {'fault_0.1': 0.13150950535987868, 'fault_0.5': 0.07299828765114973, 'fault_1.0': 0.037346351896804825, 'fault_5.0': 0.14102747959174478, 'fault_10.0': 0.1349916664094491, 'fault_20.0': 0.004974937185532994}


In [3]:
##have to do the same for the file =s in the cifar-10/cluster folder
resnet18_e = torch.load('cifar-10/cluster/resnet18_cifar10_error.pth')
vgg16_e = torch.load('cifar-10/cluster/vgg16_cifar10_error.pth')
densenet121_e= torch.load('cifar-10/cluster/densenet121_cifar10_error.pth')

In [6]:
resnet18_avg, resnet18_std = avg_std(resnet18_e)
gg16_avg, vgg16_std = avg_std(vgg16_e)
densenet121_avg, densenet121_std = avg_std(densenet121_e)
##print the avg and std
print("resnet18_avg",resnet18_avg)
print("resnet18_std",resnet18_std)
print("vgg16_avg",vgg16_avg)
print("vgg16_std",vgg16_std)
print("densenet121_avg",densenet121_avg)
print("densenet121_std",densenet121_std)

resnet18_avg {'fault_0.1': 93.07970000000003, 'fault_0.5': 87.935, 'fault_1.0': 39.7055, 'fault_5.0': 9.998}
resnet18_std {'fault_0.1': 0.3416619820817057, 'fault_0.5': 3.105213519228589, 'fault_1.0': 17.24523721930203, 'fault_5.0': 0.3390250728191059}


NameError: name 'vgg16_avg' is not defined

In [5]:
resnet18_ec = torch.load('cifar-10/resnet18_cifar10_cluster_error_ec.pth')
vgg16_ec = torch.load('cifar-10/vgg16_cifar10_cluster_error_ec.pth')
densenet121_ec= torch.load('cifar-10/densenet121_cifar10_cluster_error_ec.pth')

In [6]:
resnet18_avg, resnet18_std = avg_std(resnet18_ec)
vgg16_avg, vgg16_std = avg_std(vgg16_ec)
densenet121_avg, densenet121_std = avg_std(densenet121_ec)
##print the avg and std
print("resnet18_avg",resnet18_avg)
print("resnet18_std",resnet18_std)
print("vgg16_avg",vgg16_avg)
print("vgg16_std",vgg16_std)
print("densenet121_avg",densenet121_avg)
print("densenet121_std",densenet121_std)

resnet18_avg {'fault_0.1': 93.75800000000001, 'fault_0.5': 93.75199999999998, 'fault_1.0': 93.76500000000001, 'fault_5.0': 93.693, 'fault_10.0': 93.651, 'fault_15.0': 93.57300000000001, 'fault_20.0': 93.534, 'fault_30.0': 93.238, 'fault_40.0': 93.23299999999999, 'fault_50.0': 93.17599999999999}
resnet18_std {'fault_0.1': 0.06539113089708708, 'fault_0.5': 0.07263607918933937, 'fault_1.0': 0.06407027391856655, 'fault_5.0': 0.15258112596255174, 'fault_10.0': 0.16585837331892403, 'fault_15.0': 0.24372320365529287, 'fault_20.0': 0.21850400453996396, 'fault_30.0': 0.29952629266894265, 'fault_40.0': 0.22248820193439597, 'fault_50.0': 0.19422667170087418}
vgg16_avg {'fault_0.1': 91.79499999999999, 'fault_0.5': 91.806, 'fault_1.0': 91.795, 'fault_5.0': 91.728, 'fault_10.0': 91.695, 'fault_15.0': 91.747, 'fault_20.0': 91.687, 'fault_30.0': 91.594, 'fault_40.0': 91.631, 'fault_50.0': 91.55600000000001}
vgg16_std {'fault_0.1': 0.06682065548915261, 'fault_0.5': 0.05919459434779533, 'fault_1.0': 0.0